# Schema Registry & Serialization

## What's covered

- Why opaque bytes is a footgun at scale — the producer/consumer schema-drift problem
- The serialization-format options — plain bytes, JSON, Avro, Protobuf, JSON Schema — and where each fits
- Confluent Schema Registry — what it stores, what it does *not* store
- The Confluent wire format — the magic byte, the schema ID, why payloads are smaller than naive JSON
- Subject naming strategies — `TopicNameStrategy`, `RecordNameStrategy`, `TopicRecordNameStrategy`
- Compatibility modes — `BACKWARD`, `FORWARD`, `FULL`, `NONE`, and the `_TRANSITIVE` variants
- Schema evolution rules — what you can change without breaking, illustrated with Avro
- Avro in practice — schema definition, the Python producer/consumer pattern
- Protobuf and JSON Schema — when to pick them instead
- The Schema Registry REST API — subjects, versions, compatibility checks
- Common gotchas

## The opaque-bytes problem

Kafka brokers treat every record value as opaque bytes. That's the right design for the broker — it should not be in the business of understanding payload formats. But it pushes the entire schema problem onto producers and consumers, and "plain bytes by convention" is exactly how the failure happens.

Picture a `payments.transaction.created` topic. On day one, both producer and consumer agree the payload is JSON shaped like `{"id": str, "amount": float}`. Six months later:

- The producer team renames `amount` to `amount_cents` and changes the unit. **No consumer was notified.** Every consumer silently breaks or, worse, silently processes wrong values.
- A new consumer team writes against a year-old example payload and trips over a field that was added six months back.
- The legal team needs to know what data flows through this topic. The only authoritative answer is to read the producer source — assuming you can find it.

**Schema Registry exists to solve this socially:** schemas live in a versioned, queryable store that producers and consumers both consult, and the registry enforces *compatibility rules* on every new schema. Producers can't ship a breaking change without the registry refusing the new version.

The format you pick (Avro, Protobuf, JSON Schema) matters less than the *practice* of having a schema at all. Pick one and don't go back.

## Format comparison

Four practical choices, with the trade-offs that matter:

| Format | Wire size | Schema is required? | Evolution story | Where it shines |
|---|---|---|---|---|
| **Plain JSON** | Largest (text + field names per record) | No | None — pure convention | Quick prototypes, human-readable debugging |
| **JSON Schema** | Same as JSON | Yes, in registry | Good — but verbose | JSON-native consumers, REST-shaped systems |
| **Avro** | Small (binary, no field names per record) | Yes — required to deserialize | Excellent — designed around it | Default in the Kafka ecosystem; Hadoop/Spark interop |
| **Protobuf** | Smallest (binary, field numbers) | Yes | Excellent — field numbers, never reuse them | gRPC interop, polyglot teams |

Two things that surprise newcomers:

- **Avro records on the wire do not include field names.** Decoding requires the writer's schema. That's why Schema Registry is mandatory for Avro in Kafka — the consumer fetches the writer's schema by ID, then decodes.
- **Protobuf records don't include field names either.** They include field *numbers*. That's why "never reuse a field number" is the cardinal rule of Protobuf evolution — an old consumer reading a record with a renumbered field will misinterpret it.

**The default recommendation:** Avro if you're in the Kafka/Hadoop/Spark ecosystem; Protobuf if you already have gRPC services; JSON Schema if a chunk of your consumers will read records by hand. Plain JSON only for prototypes — once a topic has more than one consumer team, you owe yourself a schema.

## What Schema Registry actually is

Schema Registry is a small HTTP service that sits beside the Kafka cluster. Its data model is dead simple:

- **Schema** — a string (Avro JSON, Protobuf `.proto`, or JSON Schema text).
- **Subject** — a named, versioned bucket of schemas. Default mapping is one subject per `(topic, key-or-value)` pair: `payments.created-value`, `payments.created-key`.
- **Schema ID** — a global, monotonically increasing integer assigned to every unique schema across the whole registry.
- **Compatibility mode** — set per subject (or globally), the rules new schema versions must satisfy.

```text
  ┌─────────────────────────────────────────────────────────────┐
  │ Schema Registry                                             │
  │                                                             │
  │   subject: payments.created-value                           │
  │     v1 → schema-id 7   {name: str, amount: double}          │
  │     v2 → schema-id 12  {name: str, amount: double,          │
  │                         currency: str = "USD"}              │
  │                                                             │
  │   subject: users.profile-value                              │
  │     v1 → schema-id 9   ...                                  │
  └─────────────────────────────────────────────────────────────┘
             ▲                       ▲
             │                       │
        register or                 fetch by ID
        lookup by hash              when decoding
             │                       │
        producers               consumers
```

What Schema Registry does **not** do:

- It does not validate records (that's the serializer's job).
- It does not store records themselves (those still live in Kafka).
- It does not enforce that a topic's records actually match the registered schema — only the *serializer* enforces that. A misbehaving producer that bypasses the serializer can still write garbage.

## The Confluent wire format

Every record serialized through a Schema Registry serializer is laid out as:

```text
  ┌────┬────────────────┬─────────────────────────────────────┐
  │ 0  │  schema-id     │   serialized payload (Avro/Proto)   │
  │ 1B │   4 bytes BE   │     N bytes                         │
  └────┴────────────────┴─────────────────────────────────────┘
    magic    schema ID         actual encoded record
```

- **Byte 0 — magic byte.** Always `0x00`. Reserves room for future versions of the wire format.
- **Bytes 1–4 — schema ID.** Big-endian 32-bit integer. The deserializer reads this first, fetches the schema by ID from the registry (caching aggressively), then decodes the payload.
- **Bytes 5+ — payload.** The actual Avro/Protobuf/JSON-Schema bytes. No field names for Avro, no field names for Protobuf, full JSON for JSON Schema.

Two practical consequences:

- **Five bytes of overhead per record.** Negligible at any realistic record size.
- **You cannot read a Schema Registry record without the registry.** A consumer that loses connectivity to Schema Registry can't decode anything. Plan for that operationally — run the registry close to your consumers, with its own retention and backup.

## Subject naming strategies

How does a subject name get derived from a topic name? The producer's `SubjectNameStrategy` decides:

| Strategy | Subject = | When to use |
|---|---|---|
| **`TopicNameStrategy`** (default) | `<topic>-value` / `<topic>-key` | One record type per topic — the common case |
| **`RecordNameStrategy`** | The fully qualified record name (e.g. `com.example.PaymentCreated`) | Many topics share the same record type — keeps one schema reused everywhere |
| **`TopicRecordNameStrategy`** | `<topic>-<record-name>` | One topic carries multiple record types and you want each type evolved independently |

**Pick `TopicNameStrategy` unless you have a specific reason not to.** It maps cleanly onto "one topic, one schema," which is the right default for most domain event streams.

**`TopicRecordNameStrategy`** is the way to do "polymorphic topics" cleanly — e.g. an `events` topic that carries `OrderCreated`, `OrderShipped`, `OrderCancelled` records side by side, each with its own evolution history. The trade-off is that consumers must be ready to handle any of the record types they subscribe to.

## Compatibility modes — the schema-evolution contract

When you register a new version of a schema, the registry checks it against the existing versions. The compatibility mode (set per-subject or globally) decides *what* it checks:

| Mode | A new schema can be registered if... | Rolling deploy order |
|---|---|---|
| **`NONE`** | Always | No safety; avoid |
| **`BACKWARD`** | An old reader can decode records written with the new schema | **Upgrade consumers first**, then producers |
| **`FORWARD`** | A new reader can decode records written with the old schema | Upgrade producers first, then consumers |
| **`FULL`** | Both `BACKWARD` and `FORWARD` | Either order works |
| **`*_TRANSITIVE`** | The check is run against *all* previous versions, not just the most recent | Stricter; the right default in practice |

**The default is `BACKWARD`**, and it's the right one for almost every event-streaming topic. Backward-compatible means: "old consumers keep working as new producers ship the new schema." You upgrade consumers first to learn the new schema, then producers start writing it. Existing consumers continue reading without changes.

The `_TRANSITIVE` variant of any mode (e.g. `BACKWARD_TRANSITIVE`) checks compatibility against every version since v1, not just the latest. This catches "two compatible hops that aren't compatible end-to-end" — useful when very old consumer versions might still be in production.

## What's safe to change (Avro)

Under `BACKWARD` (the default), here's the practical rule of thumb for Avro:

| Change | Safe? | Notes |
|---|---|---|
| Add a field **with a default value** | ✅ | Old consumers' missing field is filled with the default |
| Remove a field that had a default | ✅ | Old consumers' default fills in for the absent field |
| Add a field **without** a default | ❌ | Old consumers cannot decode the new field |
| Remove a required (no-default) field | ❌ | Old consumers expect it; deserialization fails |
| Rename a field | ❌ | Use `aliases` instead — `"aliases": ["old_name"]` |
| Change a field's type | ⚠️ | Only certain promotions (`int → long`, `int → float`, `float → double`, `string → bytes`) |
| Add a value to a union | ⚠️ | Old consumers can't handle the new branch |
| Add a value to an enum | ❌ (BACKWARD) | Add a default to the enum schema first |

**The single most useful rule to remember:** *always give new fields a default value.* It's the cheapest way to keep evolution safe, and it makes both backward and forward compatibility almost automatic.

## Setup

Schema Registry is a Confluent component, not part of `apache/kafka`. The simplest path is the Confluent image, pointed at the same broker:

```bash
docker run -d --name schema-registry --network host \
    -e SCHEMA_REGISTRY_HOST_NAME=localhost \
    -e SCHEMA_REGISTRY_LISTENERS=http://0.0.0.0:8081 \
    -e SCHEMA_REGISTRY_KAFKASTORE_BOOTSTRAP_SERVERS=PLAINTEXT://localhost:9092 \
    confluentinc/cp-schema-registry:7.7.1
```

Python dependencies (from the repo's `CLAUDE.md`):

```bash
pip install "confluent-kafka[avro]==2.5.3" fastavro==1.9.4 requests==2.32.3
```

The cells below assume both Kafka (`localhost:9092`) and Schema Registry (`localhost:8081`) are running.

In [ ]:
from confluent_kafka import Producer, Consumer
from confluent_kafka.admin import AdminClient, NewTopic
from confluent_kafka.schema_registry import SchemaRegistryClient, Schema
from confluent_kafka.schema_registry.avro import AvroSerializer, AvroDeserializer
from confluent_kafka.serialization import StringSerializer, StringDeserializer, SerializationContext, MessageField
import requests

BOOTSTRAP = "localhost:9092"
SR_URL = "http://localhost:8081"
TOPIC = "payments.created"

admin = AdminClient({"bootstrap.servers": BOOTSTRAP})
try:
    fut = admin.create_topics([NewTopic(TOPIC, num_partitions=3, replication_factor=1)])[TOPIC]
    fut.result(); print(f"created {TOPIC}")
except Exception as e:
    print(f"{TOPIC}: {e}")

sr = SchemaRegistryClient({"url": SR_URL})
print("schema registry reachable:", requests.get(f"{SR_URL}/subjects", timeout=2).status_code == 200)

## Avro — produce with v1 of the schema

Define a small Avro schema for `payments.created` and produce a few records through the Schema Registry serializer. On first produce, the serializer registers the schema (if it isn't already there) and stamps every record with the assigned schema ID.

In [ ]:
schema_v1 = """
{
  "type": "record",
  "namespace": "com.example.payments",
  "name": "PaymentCreated",
  "fields": [
    {"name": "payment_id", "type": "string"},
    {"name": "customer_id", "type": "string"},
    {"name": "amount", "type": "double"}
  ]
}
"""

key_ser = StringSerializer()
val_ser = AvroSerializer(sr, schema_v1)

producer = Producer({"bootstrap.servers": BOOTSTRAP, "acks": "all", "enable.idempotence": True})

v1_records = [
    {"payment_id": "P-001", "customer_id": "CUST0001", "amount": 120.00},
    {"payment_id": "P-002", "customer_id": "CUST0002", "amount":  40.50},
    {"payment_id": "P-003", "customer_id": "CUST0001", "amount":  75.25},
]

for r in v1_records:
    producer.produce(
        topic=TOPIC,
        key=key_ser(r["customer_id"], SerializationContext(TOPIC, MessageField.KEY)),
        value=val_ser(r, SerializationContext(TOPIC, MessageField.VALUE)),
    )
producer.flush()
print("produced 3 v1 records")

## Avro — evolve the schema to v2 (backward-compatible)

Add a `currency` field with a default of `"USD"`. This is the *canonical* safe Avro evolution — old consumers reading new records get `"USD"` filled in for the missing field, new consumers reading old records do the same.

Producing with v2 below registers a new version under the same subject. The Schema Registry validates that v2 is backward-compatible with v1; if it isn't, the call raises an error before any record ships.

In [ ]:
schema_v2 = """
{
  "type": "record",
  "namespace": "com.example.payments",
  "name": "PaymentCreated",
  "fields": [
    {"name": "payment_id",  "type": "string"},
    {"name": "customer_id", "type": "string"},
    {"name": "amount",      "type": "double"},
    {"name": "currency",    "type": "string", "default": "USD"}
  ]
}
"""

val_ser_v2 = AvroSerializer(sr, schema_v2)

v2_records = [
    {"payment_id": "P-004", "customer_id": "CUST0003", "amount":  500.00, "currency": "USD"},
    {"payment_id": "P-005", "customer_id": "CUST0004", "amount": 1200.00, "currency": "EUR"},
]

for r in v2_records:
    producer.produce(
        topic=TOPIC,
        key=key_ser(r["customer_id"], SerializationContext(TOPIC, MessageField.KEY)),
        value=val_ser_v2(r, SerializationContext(TOPIC, MessageField.VALUE)),
    )
producer.flush()
print("produced 2 v2 records — backward-compatible evolution accepted")

## Avro — consume both versions through one schema

The consumer reads back all five records using a single deserializer. Because v2 is backward-compatible with v1, the deserializer can decode both old and new records into the v2 shape — old records get `currency="USD"` filled in automatically.

This is the whole point of schema evolution: consumers don't have to know about every historical version separately.

In [ ]:
# AvroDeserializer with no reader_schema → use the writer schema (referenced by the
# embedded ID); pass schema_v2 to force projection onto v2.
key_de = StringDeserializer()
val_de = AvroDeserializer(sr, schema_v2)

c = Consumer({
    "bootstrap.servers": BOOTSTRAP,
    "group.id": "sr-demo-reader-v2",
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
})
c.subscribe([TOPIC])

print(f"{'key':<10}  payment    amount     currency")
print("-" * 45)
read = 0
while read < 5:
    msg = c.poll(2.0)
    if msg is None: break
    if msg.error(): continue
    key = key_de(msg.key(), SerializationContext(TOPIC, MessageField.KEY))
    val = val_de(msg.value(), SerializationContext(TOPIC, MessageField.VALUE))
    print(f"{key:<10}  {val['payment_id']:<8}  {val['amount']:>8.2f}  {val['currency']}")
    read += 1
c.close()

## What a *breaking* evolution looks like

Try to register a v3 that removes the required `payment_id` field. Under the default `BACKWARD` mode, the registry should reject it — an old consumer expecting `payment_id` cannot decode a v3 record where the field is gone.

We catch the registry's error and print it; no records are produced.

In [ ]:
schema_v3_breaking = """
{
  "type": "record",
  "namespace": "com.example.payments",
  "name": "PaymentCreated",
  "fields": [
    {"name": "customer_id", "type": "string"},
    {"name": "amount",      "type": "double"},
    {"name": "currency",    "type": "string", "default": "USD"}
  ]
}
"""

# Pre-flight compatibility check via the REST API. The registry returns
# {"is_compatible": true/false}; if false, the producer would fail at first
# produce attempt.
subject = f"{TOPIC}-value"
check = requests.post(
    f"{SR_URL}/compatibility/subjects/{subject}/versions/latest",
    headers={"Content-Type": "application/vnd.schemaregistry.v1+json"},
    json={"schemaType": "AVRO", "schema": schema_v3_breaking},
    timeout=5,
).json()
print("compatibility check on v3:", check)

## The Schema Registry REST API — what's there

Schema Registry is just HTTP. Every operation you'll do from a client is a REST call you can hit with `curl`:

| Endpoint | What it does |
|---|---|
| `GET /subjects` | List every subject in the registry |
| `GET /subjects/{subject}/versions` | List the version numbers for a subject |
| `GET /subjects/{subject}/versions/{v}` | Fetch a specific schema version (`latest` works too) |
| `GET /schemas/ids/{id}` | Fetch a schema by its global ID — what deserializers do internally |
| `POST /subjects/{subject}/versions` | Register a new schema version (the serializer does this for you on first produce) |
| `POST /compatibility/subjects/{subject}/versions/{v}` | Check whether a candidate schema is compatible — without registering it |
| `PUT /config` / `PUT /config/{subject}` | Set the global or per-subject compatibility mode |
| `GET /config/{subject}` | Read the effective compatibility mode |
| `DELETE /subjects/{subject}` | Soft-delete a subject (recoverable); add `?permanent=true` to hard-delete |

Below: list the subjects we've registered, then fetch the registered v2 of `payments.created-value`.

In [ ]:
subjects = requests.get(f"{SR_URL}/subjects", timeout=5).json()
print("subjects:", subjects)

versions = requests.get(f"{SR_URL}/subjects/{subject}/versions", timeout=5).json()
print(f"versions of {subject}: {versions}")

latest = requests.get(f"{SR_URL}/subjects/{subject}/versions/latest", timeout=5).json()
print(f"latest schema id : {latest['id']}")
print(f"latest version   : {latest['version']}")

cfg = requests.get(f"{SR_URL}/config/{subject}", timeout=5)
# 404 if no per-subject override — fall back to global
if cfg.status_code == 404:
    cfg = requests.get(f"{SR_URL}/config", timeout=5)
print("compatibility mode:", cfg.json())

## Protobuf and JSON Schema

The pattern with `confluent-kafka` is the same; only the serializer changes.

**Protobuf.** Define a `.proto`, generate Python bindings with `protoc`, then:

```python
from confluent_kafka.schema_registry.protobuf import ProtobufSerializer, ProtobufDeserializer
from my_proto import PaymentCreated_pb2

val_ser = ProtobufSerializer(PaymentCreated_pb2.PaymentCreated, sr, {"use.deprecated.format": False})
```

Evolution rules: never reuse a field number, mark removed fields with `reserved`, give new fields explicit numbers. Compatibility checks work the same as Avro.

**JSON Schema.** Schema is a JSON-Schema document; payload is regular JSON. Closest to plain JSON in feel, with the registry enforcing structure:

```python
from confluent_kafka.schema_registry.json_schema import JSONSerializer, JSONDeserializer

schema = '''{"type": "object", "properties": {...}, "required": [...]}'''
val_ser = JSONSerializer(schema, sr)
```

JSON Schema produces records you can `cat | jq` if you strip the five-byte header — useful for debugging. Wire size is the largest of the three; pick it when human-readability outweighs efficiency.

## Common gotchas

- **Forgetting the magic byte.** Trying to deserialize a raw Avro/Protobuf payload (no Confluent header) with a Schema Registry deserializer fails. The five-byte prefix is non-optional in the Kafka ecosystem.
- **Bypassing the serializer in one producer.** All it takes is one rogue producer writing plain JSON to a Schema-Registry-managed topic to corrupt consumer pipelines. Enforce serializer use in code review or via broker-side schema-validation (Confluent server-side schema validation does this).
- **Wrong compatibility mode for the rollout order.** `BACKWARD` requires consumers upgraded first; `FORWARD` requires producers first. Pick the mode that matches how your team actually deploys.
- **Renaming a field without an alias.** Treated as remove-old + add-new; both break `BACKWARD` compatibility. Use `"aliases": ["old_name"]` instead.
- **Adding a no-default field.** The single most common breaking change in Avro topics. Always add a default.
- **Schema Registry as a single point of failure.** Consumers cannot decode anything without it. Run it HA, with the `_schemas` backing topic replicated, and monitor for unavailability.
- **One Schema Registry per environment.** Schemas are environment-scoped — dev and prod registries are independent. Don't try to share.
- **Soft-delete then re-register a different schema.** Subject IDs survive soft-deletes; the registry may reject the new schema for compatibility against the deleted version. Hard-delete (`?permanent=true`) if you really want to reset.

## What's next

Records now have a shape the system can enforce. Producers can't ship breaking changes; consumers don't have to guess the wire format.

- **Notebook 06 — Kafka Connect.** Pre-built source and sink connectors, running as long-lived workers — so a CDC pipeline from Postgres or a sink to S3 is config, not code. Schema Registry plugs in here too: the JDBC source connector emits Avro records keyed by table schema.
- **Notebook 07 — Kafka Streams.** Stateful stream processing — joins, aggregations, windowed counts — over the schematized topics you can now trust.
- **Notebook 08 — Operations, Security & Performance Tuning.** Closes the curriculum with the operational dials: monitoring, SSL/SASL/ACLs, broker tuning.